In [ ]:
!pip install numpy pandas torch scikit-learn tensorflow deepctr-torch

In [ ]:
#################################
# Import Libraries
#################################
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from deepctr_torch.inputs import SparseFeat, get_feature_names
from deepctr_torch.models import DeepFM
import matplotlib.pyplot as plt

In [ ]:
#################################
# Load and Preprocess Data
#################################
# Load and preprocess data
data = pd.read_json("experiment_1/Amazon_Fashion.jsonl", lines=True)
data = data[data["rating"] != 3]
data = data[data["timestamp"] > '2015-01-01']
data["year"] = data["timestamp"].apply(lambda x: x.year)
data["month"] = data["timestamp"].apply(lambda x: x.month)
data["day"] = data["timestamp"].apply(lambda x: x.day)
data["rating"] = data["rating"].transform(lambda x: 1 if x > 3 else 0)


# print length of data
print(len(data))

# Set index to timestamp
data = data.set_index("timestamp")
data.sort_index(inplace=True)

In [ ]:
#################################
# Define Ground Truth Functionality
#################################
# Calculate rolling mean and threshold
rolling_mean = data["rating"].rolling("30D").mean()
rolling_std = data["rating"].rolling("30D").std()
significance_threshold = rolling_mean - 1.96 * rolling_std

# Add ground truth drift flag based on threshold
data["drift_flag"] = (data["rating"] < significance_threshold).fillna(False)


#################################
# Visualize Data and Mark Observed Drift
#################################
plt.figure(figsize=(14, 7))

# Plot ratings over time
plt.plot(data.index, data["rating"], label="Ratings", alpha=0.5)
# Plot rolling mean
plt.plot(data.index, rolling_mean, label="Rolling Mean (30D)", color="orange")
# Plot significance threshold
plt.plot(data.index, significance_threshold, label="Threshold (Mean - 1.96*Std)", color="red", linestyle="--")

# Highlight drift periods
plt.fill_between(
    data.index,
    rolling_mean,
    significance_threshold,
    where=data["drift_flag"],
    color="red",
    alpha=0.2,
    label="Drift Detected",
)

# Highlight observed drift (2021–2023)
plt.axvspan(
    pd.Timestamp("2021-01-01"), 
    pd.Timestamp("2023-01-01"), 
    color="purple", 
    alpha=0.1, 
    label="Observed Drift (2021-2023)"
)

# Add titles and legend
plt.title("Ratings Over Time with Drift Detection", fontsize=16)
plt.xlabel("Timestamp", fontsize=12)
plt.ylabel("Rating", fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
# print data features
print("Data Features:" + str(data.columns))

print(f"length of data: {len(data)}")

sparse_features = ["asin", "parent_asin", "user_id"]
target = "rating"

print("data_flag: " + str(data["drift_flag"].value_counts()))
# 
# print(data["asin"].nunique())  # Should show high cardinality
# print(data["user_id"].nunique())  # Should show high cardinality

print("asin: " + str(data["asin"].nunique()))
print("user_id: " + str(data["user_id"].nunique()))


In [ ]:
#################################
# Utility Functions
#################################
def compute_mahalanobis_distance(mean, cov_inv, embedding):
    diff = embedding - mean
    return np.sqrt(diff.T @ cov_inv @ diff)

class DriftDetector:
    def __init__(self, embedding_dim, alpha=0.01):
        self.alpha = alpha
        self.mean = np.zeros(embedding_dim)
        self.cov = np.eye(embedding_dim)
        self.count = 0

    def update(self, embedding):
        if self.count == 0:
            self.mean = embedding
            self.cov = np.eye(len(embedding))
        else:
            self.mean = (1 - self.alpha) * self.mean + self.alpha * embedding
            diff = embedding - self.mean
            self.cov = (1 - self.alpha) * self.cov + self.alpha * np.outer(diff, diff)
        self.count += 1

    def detect_drift(self, embedding):
        return compute_mahalanobis_distance(self.mean, np.linalg.pinv(self.cov), embedding)

#################################
# Model Training (DeepFM)
#################################
# Label encoding
for feat in sparse_features:
    lbe = LabelEncoder()
    data[feat] = lbe.fit_transform(data[feat])

# Define feature columns
embedding_dim = 64
fixlen_feature_columns = [
    SparseFeat(feat, data[feat].nunique(), embedding_dim=embedding_dim)
    for feat in sparse_features
]
linear_feature_columns = fixlen_feature_columns
dnn_feature_columns = fixlen_feature_columns
feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

# Prepare model input
model_input = {name: data[name] for name in sparse_features}

# Decide on device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    
model = DeepFM(linear_feature_columns, dnn_feature_columns, task='binary', device=device)
model.compile("adam", metrics=["AUC"], loss='binary_crossentropy')

# Train the model (using some fraction of data as data_sample)
data_sample = data.sample(frac=1.0, random_state=42)  # example sampling
history = model.fit(
    {name: data_sample[name] for name in sparse_features},
    data_sample[target].values,
    batch_size=1024,
    epochs=2,
    verbose=2,
    validation_split=0.1
)

In [ ]:
import numpy as np
import torch
from sklearn.metrics import confusion_matrix

#################################
# Mahalanobis Drift Detection Setup
#################################
window_size = 50
threshold_multiplier = 3.0
alpha = 0.01

tracker = DriftDetector(embedding_dim, alpha=alpha)
drift_distances = []
thresholds = []
predicted_drift_labels = []

# To map each batch back to its ground truth drift label, we need indexing:
n_batches = 300
batch_list = np.array_split(data_sample, n_batches)

with torch.no_grad():
    for batch_index, batch_df in enumerate(batch_list):
        # print(f"\n>>> Debugging: Processing Batch {batch_index}")
        # Prepare batch input
        batch_input = {key: batch_df[key].values for key in sparse_features}
        
        # Get model's embeddings or outputs
        embeddings = model.predict(batch_input)
        mean_embedding = np.mean(embeddings, axis=0)
        
        # Update and compute distance
        tracker.update(mean_embedding)
        drift_distance = tracker.detect_drift(mean_embedding)
        drift_distances.append(drift_distance)
        
        # Dynamic threshold
        if len(drift_distances) > window_size:
            recent_dists = drift_distances[-window_size:]
            mean_dist = np.mean(recent_dists)
            std_dist = np.std(recent_dists)
            threshold = mean_dist + threshold_multiplier * std_dist
        else:
            threshold = (
                np.mean(drift_distances) +
                threshold_multiplier * np.std(drift_distances)
            )
        thresholds.append(threshold)
        
        # Predicted drift (batch level)
        if drift_distance > threshold:
            predicted_drift_labels.append(1)
        else:
            predicted_drift_labels.append(0)

        # Debugging: Print out each batch's drift distance and threshold
        # print(
        #     f"--- Debugging: drift_distance = {drift_distance:.4f}, "
        #     f"threshold = {threshold:.4f}, "
        #     f"predicted_label = {predicted_drift_labels[-1]}"
        # )

        # If drift is detected
        if predicted_drift_labels[-1] == 1:
            print(f"Batch {batch_index}: Drift detected! Distance = {drift_distance:.4f}, Threshold = {threshold:.4f}")


plt.plot(drift_distances, label="Drift Distance")
plt.plot(thresholds, label="Threshold")
plt.title("Mahalanobis Drift Detection")
plt.xlabel("Batch Index")
plt.ylabel("Distance")
plt.legend()
plt.show()


In [ ]:
#################################
# Compute Ground Truth (Batch-Level) and FPR
#################################
# Ground truth drift label per batch: >X% drift_flag -> drift
# This is based on the imbalance observed in the drift_flag feature.
actual_drift_labels = []
for i, batch_df in enumerate(batch_list):
    batch_flags = batch_df["drift_flag"]
    drift_fraction = batch_flags.mean()
    label = 1 if drift_fraction > 0.06 else 0
    actual_drift_labels.append(label)

# Convert to numpy arrays
y_true = np.array(actual_drift_labels)
y_pred = np.array(predicted_drift_labels)

# Debugging: Check if lengths match
assert len(y_true) == len(y_pred), f"y_true and y_pred lengths do not match: {len(y_true)} vs {len(y_pred)}"

# Compute confusion matrix
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

# Avoid division by zero in FPR calculation
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

# Print results
print("\nConfusion Matrix:")
print(f"  TP={tp}, FN={fn}, FP={fp}, TN={tn}")
print(f"False Positive Rate (FPR): {fpr:.4f}")

#################################
# Save Results
#################################
distances_path = "drift_distances_dl.npy"
thresholds_path = "drift_thresholds_dl.npy"
np.save(distances_path, drift_distances)
np.save(thresholds_path, thresholds)
print("\nDrift detection results saved.")